# Week 2 — Featurization + Baseline Models

Turns each molecule's SMILES string into a numeric feature vector a model can consume, then trains a baseline random forest per property.

Two representations, combined:
- **Morgan fingerprints** — circular substructure fingerprints, the standard cheminformatics baseline. Captures fine-grained structural detail but isn't human-interpretable.
- **Lipinski-style descriptors** (MolWt, LogP, HBD, HBA, rotatable bonds) — coarser, but each one has a direct chemical meaning, which is useful later for feature-importance interpretation (Week 3).

Reuses `largest_fragment()` from `Data_EDA.ipynb` so salt-stripping stays consistent with the EDA pass.

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.DataStructs import ConvertToNumpyArray


# Strips counter-ions/solvents from a salt, keeping only the largest fragment
# (by heavy-atom count) — e.g. Clofilium phosphate -> just the clofilium cation.
# Reused from Data_EDA.ipynb so salt-stripping stays consistent with the EDA pass.
def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) == 1:
        return mol
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())


# Encodes a molecule's local substructures as a fixed-length bit vector, so a
# molecule of any size/shape becomes a feature array of constant length (n_bits).
def compute_morgan_fp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr

In [2]:
# Converts a whole dataframe of SMILES into one feature matrix ready for
# scikit-learn: fingerprint bits + interpretable descriptors, side by side.
def featurize(df, smiles_col='Drug', radius=2, n_bits=2048):
    fp_rows = []
    desc_rows = []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(smi)
        mol = largest_fragment(mol)
        fp_rows.append(compute_morgan_fp(mol, radius=radius, n_bits=n_bits))
        desc_rows.append({
            'MolWt': Descriptors.MolWt(mol),
            'MolLogP': Descriptors.MolLogP(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        })

    fp_cols = [f'fp_{i}' for i in range(n_bits)]
    fp_df = pd.DataFrame(np.vstack(fp_rows), columns=fp_cols, index=df.index)
    desc_df = pd.DataFrame(desc_rows, index=df.index)
    return pd.concat([fp_df, desc_df], axis=1)

## Solubility — baseline model

Proving the pipeline end-to-end on the cleanest dataset first (no missing data, no label conflicts). Load via TDC's **scaffold split**, featurize, train a baseline random forest, evaluate.

In [3]:
from tdc.single_pred import ADME

# Scaffold split groups molecules by their core ring structure (scaffold) before
# splitting, so train and test contain structurally different molecules — a
# harder, more realistic generalization test than a random split, and how the
# TDC leaderboard itself evaluates.
solubility = ADME(name='Solubility_AqSolDB')
scaffold_split = solubility.get_split(method='scaffold')

train_df = scaffold_split['train']
test_df = scaffold_split['test']
print(f"train: {train_df.shape[0]} molecules, test: {test_df.shape[0]} molecules")

Found local copy...


Loading...


Done!


  0%|          | 0/9982 [00:00<?, ?it/s]

 12%|█▏        | 1218/9982 [00:00<00:00, 11726.66it/s]

...

 67%|██████▋   | 6649/9982 [00:00<00:00, 12883.66it/s]

 82%|████████▏ | 8148/9982 [00:00<00:00, 13558.01it/s]

 96%|█████████▌| 9542/9982 [00:00<00:00, 13235.12it/s]

100%|██████████| 9982/9982 [00:00<00:00, 12180.03it/s]

train: 6987 molecules, test: 1997 molecules


In [4]:
# Featurize both splits with the function defined above (fingerprints + descriptors).
# This is the slow step — computing a 2048-bit fingerprint per molecule.
X_train = featurize(train_df)
X_test = featurize(test_df)

y_train = train_df['Y'].values
y_test = test_df['Y'].values

X_train.shape, X_test.shape

((6987, 2053), (1997, 2053))

In [5]:
from sklearn.ensemble import RandomForestRegressor

# Baseline model: default hyperparameters aside from n_estimators, so we get an
# honest "no tuning" reference point before Week 3's tuning/comparison pass.
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Evaluate on the held-out scaffold-split test set — molecules structurally
# unseen during training. RMSE and MAE are in the same units as Y (log
# solubility); R^2 is the fraction of variance in Y the model explains
# (1.0 = perfect). MAE is included because it's the metric the TDC
# leaderboard itself reports for this dataset, so it's directly comparable.
y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R^2:  {r2:.3f}")

RMSE: 1.286
MAE:  0.926
R^2:  0.686


## BBB Permeability — baseline model

Reusing the same `featurize()` pipeline, but this is a **classification** task (permeable vs. not), so the model and metrics change: `RandomForestClassifier` instead of `Regressor`, and ROC-AUC instead of RMSE/R² — chosen back in Week 1 EDA because the classes are imbalanced (77% permeable / 23% not), which makes raw accuracy misleading.

In [7]:
from tdc.single_pred import ADME

# Same scaffold-split logic as Solubility: structurally distinct train/test.
bbb = ADME(name='BBB_Martins')
bbb_scaffold_split = bbb.get_split(method='scaffold')

bbb_train_df = bbb_scaffold_split['train']
bbb_test_df = bbb_scaffold_split['test']
print(f"train: {bbb_train_df.shape[0]} molecules, test: {bbb_test_df.shape[0]} molecules")
print(bbb_train_df['Y'].value_counts(normalize=True))

Found local copy...


Loading...


Done!


  0%|          | 0/2030 [00:00<?, ?it/s]

 40%|███▉      | 811/2030 [00:00<00:00, 8105.70it/s]

 80%|███████▉  | 1622/2030 [00:00<00:00, 7725.32it/s]

100%|██████████| 2030/2030 [00:00<00:00, 7880.64it/s]

train: 1421 molecules, test: 406 molecules
Y
1    0.743139
0    0.256861
Name: proportion, dtype: float64


In [8]:
# Featurize both splits — same function as Solubility, no changes needed since
# it works off SMILES + salt-stripping regardless of the target type.
X_bbb_train = featurize(bbb_train_df)
X_bbb_test = featurize(bbb_test_df)

y_bbb_train = bbb_train_df['Y'].values
y_bbb_test = bbb_test_df['Y'].values

X_bbb_train.shape, X_bbb_test.shape

((1421, 2053), (406, 2053))

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Classifier instead of regressor. class_weight='balanced' re-weights the
# minority class (non-permeable, 23%) during training so the model doesn't
# just learn to predict "permeable" for everything and still look accurate.
rf_bbb = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_bbb.fit(X_bbb_train, y_bbb_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score

# predict_proba gives P(class=1) per molecule, needed for ROC-AUC (which
# measures ranking quality across all thresholds, not just one cutoff).
# predict gives the hard 0/1 label, needed for F1 and balanced accuracy.
y_bbb_proba = rf_bbb.predict_proba(X_bbb_test)[:, 1]
y_bbb_pred = rf_bbb.predict(X_bbb_test)

roc_auc = roc_auc_score(y_bbb_test, y_bbb_proba)
f1 = f1_score(y_bbb_test, y_bbb_pred)
bal_acc = balanced_accuracy_score(y_bbb_test, y_bbb_pred)

print(f"ROC-AUC:           {roc_auc:.3f}")
print(f"F1:                {f1:.3f}")
print(f"Balanced accuracy: {bal_acc:.3f}")

ROC-AUC:           0.904
F1:                0.931
Balanced accuracy: 0.762
